In [ ]:
"""
코랩용 SFT (Supervised Fine-Tuning) 학습 스크립트
./finetuning/finetuning_data_dpo/crm-sft-dataset/cycle_01.jsonl 파일로 1 사이클 SFT 학습을 수행합니다.
./finetuning/checkpoints_sft에 Trainer 메타 데이터를 저장하고 resume을 통해 추가 학습할 수 있도록 합니다.
adapter는 /content/drive/MyDrive/멋사/adapters_sft_1에 저장합니다.
"""

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install datasets peft trl bitsandbytes accelerate
!pip install -U transformers
!pip show transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 49.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and fu

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', 'sample_data']


In [5]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
%cd AmoRe_crm_generator
!git checkout jinhyeok
!git branch
os.chdir("/content/AmoRe_crm_generator")
print(os.getcwd())

Cloning into 'AmoRe_crm_generator'...
remote: Enumerating objects: 613, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 613 (delta 73), reused 83 (delta 66), pack-reused 508 (from 1)
Receiving objects: 100% (613/613), 5.54 MiB | 18.47 MiB/s, done.
Resolving deltas: 100% (359/359), done.
/content/AmoRe_crm_generator
Branch 'jinhyeok' set up to track remote branch 'jinhyeok' from 'origin'.
Switched to a new branch 'jinhyeok'
* jinhyeok
  main
/content/AmoRe_crm_generator


In [9]:
from dotenv import load_dotenv
load_dotenv()

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [14]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

# 모델 및 경로 설정
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
CACHE_DIR = "./models"
OUTPUT_DIR = "./finetuning/checkpoints_sft_v2"
OUTPUT_ADAPTER_DIR = "/content/drive/MyDrive/LikeLion/Small Challenge/adapters_sft_1_v4"
# BASE_ADAPTER_PATH = "/content/drive/MyDrive/LikeLion/adapters_sft_base"
NEW_ADAPTER_NAME = "sft_adapter_v1"

# 데이터셋 경로 설정
# DATA_DIR = "./finetuning/finetuning_data/crm-sft-dataset"
DATA_DIR = "/content/drive/MyDrive/LikeLion/Small Challenge/dataset_sft"
JSONL_FILE = os.path.join(DATA_DIR, "cycle_01_v3.jsonl")

# 하이퍼파라미터 설정
MAX_SEQ_LENGTH = 1512


def load_sft_dataset(jsonl_path: str):
    """JSONL 파일에서 SFT 형식의 데이터셋을 로드합니다.

    JSONL 형식:
      { "prompt": "...", "chosen": "..." }

    Args:
        jsonl_path: JSONL 파일 경로

    Returns:
        train_dataset, eval_dataset
    """
    dataset = load_dataset(
        "json",
        data_files=jsonl_path,
    )
    dataset = dataset["train"]
    dataset = dataset.map(
        lambda x: {"text": x["prompt"] + "\n" + x["chosen"]},
        remove_columns=dataset.column_names,
    )

    # train / eval split
    dataset = dataset.train_test_split(test_size=0.1, seed=42)

    return dataset["train"], dataset["test"]


def _freeze_all_params(model):
    for _, param in model.named_parameters():
        param.requires_grad = False


def _enable_adapter_params(model, adapter_name):
    for name, param in model.named_parameters():
        if f".{adapter_name}." in name:
            param.requires_grad = True

def formatting_func(example):
    return example["text"]

In [15]:
"SFT 학습 메인 함수"

# 1. 토크나이저 로드
print("토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
)

# pad_token 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 패딩/트렁케이션 사이드 설정 (SFT 학습 기본)
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

# max_length 설정
tokenizer.model_max_length = MAX_SEQ_LENGTH

# 2. 데이터셋 로드
print(f"데이터셋 로드 중: {JSONL_FILE}")
if not os.path.exists(JSONL_FILE):
    raise FileNotFoundError(f"데이터셋 파일을 찾을 수 없습니다: {JSONL_FILE}")

train_dataset, eval_dataset = load_sft_dataset(JSONL_FILE)
print(f"학습 데이터: {len(train_dataset)}개, 평가 데이터: {len(eval_dataset)}개")

# 3. Flash Attention 설정
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# 4. 모델 로드
print("모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    use_cache=False,
    # attn_implementation=attn_implementation,
    torch_dtype=torch_dtype,
    cache_dir=CACHE_DIR,
)

# 5. PEFT (LoRA) 설정
print("PEFT 설정 중...")
peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.05,
    r=32,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM"
)

# 6. 베이스 어댑터 로드 (학습하지 않음)
# print(f"베이스 어댑터 로드 중: {BASE_ADAPTER_PATH}")
# if not os.path.exists(BASE_ADAPTER_PATH):
#     raise FileNotFoundError(f"베이스 어댑터를 찾을 수 없습니다: {BASE_ADAPTER_PATH}")

# model = PeftModel.from_pretrained(
#     model,
#     BASE_ADAPTER_PATH,
#     is_trainable=False,
# )

# 7. 추가 어댑터 생성 및 활성화
print(f"추가 어댑터 생성: {NEW_ADAPTER_NAME}")
model.add_adapter(peft_config, NEW_ADAPTER_NAME)
model.set_adapter(NEW_ADAPTER_NAME)
_freeze_all_params(model)
_enable_adapter_params(model, NEW_ADAPTER_NAME)

# 8. SFT Config 설정
print("SFT Config 설정 중...")
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=3,
    learning_rate=1e-4,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=1,
    logging_first_step=True,
    logging_strategy="steps",
    log_level="info",
    disable_tqdm=False,
    save_steps=100,
    save_total_limit=20,
    eval_strategy="steps",
    eval_steps=20,
    report_to="none"
)

# 9. SFTTrainer 초기화
print("SFTTrainer 초기화 중...")
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

# 10. 학습 시작
print("학습 시작...")
ckpt_dir = OUTPUT_DIR

resume = None
if os.path.isdir(ckpt_dir) and len(os.listdir(ckpt_dir)) > 0:
    resume = True

trainer.train(resume_from_checkpoint=resume)

# 11. 모델 저장
print("모델 저장 중...")
trainer.save_model(OUTPUT_ADAPTER_DIR)
print(f"모델이 저장되었습니다: {OUTPUT_ADAPTER_DIR}")


토크나이저 로드 중...


loading configuration file config.json from cache at ./models/models--LGAI-EXAONE--EXAONE-4.0-1.2B/snapshots/3abf2810673c7c0778df64a73c2d52eab32d91c4/config.json
Model config Exaone4Config {
  "architectures": [
    "Exaone4ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 361,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

데이터셋 로드 중: /content/drive/MyDrive/LikeLion/Small Challenge/dataset_sft/cycle_01_v3.jsonl
학습 데이터: 1870개, 평가 데이터: 208개
모델 로드 중...


loading configuration file config.json from cache at ./models/models--LGAI-EXAONE--EXAONE-4.0-1.2B/snapshots/3abf2810673c7c0778df64a73c2d52eab32d91c4/config.json
Model config Exaone4Config {
  "architectures": [
    "Exaone4ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 361,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

...

모델 저장 중...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--LGAI-EXAONE--EXAONE-4.0-1.2B/snapshots/3abf2810673c7c0778df64a73c2d52eab32d91c4/config.json
Model config Exaone4Config {
  "architectures": [
    "Exaone4ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 361,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attenti

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /content/drive/MyDrive/LikeLion/Small Challenge/adapters_sft_1_v4/adapter_model.safetensors
chat template saved in /content/drive/MyDrive/LikeLion/Small Challenge/adapters_sft_1_v4/chat_template.jinja
tokenizer config file saved in /content/drive/MyDrive/LikeLion/Small Challenge/adapters_sft_1_v4/tokenizer_config.json


모델이 저장되었습니다: /content/drive/MyDrive/LikeLion/Small Challenge/adapters_sft_1_v4


In [ ]:
!pip install huggingface-hub

In [ ]:
# Push to HuggingFace Hub

import os

from dotenv import load_dotenv
from huggingface_hub import login, create_repo, upload_folder

login(os.getenv("HUGGINGFACE_API_KEY"))

create_repo(
    repo_id="crm-sft-adapter-v2",
    repo_type="model",
    private=False,
    exist_ok=True
)

upload_folder(
    folder_path=OUTPUT_ADAPTER_DIR,
    repo_id="jinn33/crm-sft-adapter-v2",
    repo_type="model",
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          |  622kB /  122MB            

  ...ft_1_v2/training_args.bin:   1%|1         |  70.0B / 6.22kB            

CommitInfo(commit_url='https://huggingface.co/jinn33/crm-sft-adapter-v2/commit/fae4a0dcd6e156a1a89e4ddb8d4158f917d7f5cd', commit_message='Upload folder using huggingface_hub', commit_description='', oid='fae4a0dcd6e156a1a89e4ddb8d4158f917d7f5cd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jinn33/crm-sft-adapter-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='jinn33/crm-sft-adapter-v2'), pr_revision=None, pr_num=None)